## 一、匯入套件與cleaned data

In [83]:
# 導入必要的數據科學套件
# pandas (pd): 用於數據操作、分析和讀取各種數據格式（如CSV、Excel）
# numpy (np): 用於科學計算和數組操作
import pandas as pd
import numpy as np

# 導入繪圖庫
import matplotlib.pyplot as plt  # 用於靜態繪圖
import seaborn as sns  # 基於 Matplotlib 的統計圖形庫
import plotly.express as px  # 用於動態交互式繪圖

In [84]:
# 匯入cleaned data
df_ac = pd.read_parquet('cleaned_data/ac_普通.parquet')
df_putong1 = pd.read_parquet('cleaned_data/putong1_普通新設一.parquet')
df_putong2 = pd.read_parquet('cleaned_data/putong2_普通新設二.parquet')
df_elevator = pd.read_parquet('cleaned_data/elevator_普通電梯.parquet')
df_gongtong = pd.read_parquet('cleaned_data/gongtong_共同教室.parquet')
df_boya1 = pd.read_parquet('cleaned_data/boya1_博雅館一.parquet')
df_boya2 = pd.read_parquet('cleaned_data/boya2_博雅館二.parquet')
df_boya3 = pd.read_parquet('cleaned_data/boya3_博雅三.parquet')
df_boya4 = pd.read_parquet('cleaned_data/boya3_博雅三.parquet')
df_xinsheng = pd.read_parquet('cleaned_data/xinsheng_新生大樓.parquet')
weather = pd.read_csv('cleaned_data/weather_data_2016_2025.csv')
                              

## 二、普通分析

### 1.合併電表與溫度資料

In [85]:
# 合併電表與溫度
putong_all = pd.concat([
    df_ac[['kw']].rename(columns={'kw': 'ac'}),
    df_putong1[['kw']].rename(columns={'kw': 'm1'}),
    df_putong2[['kw']].rename(columns={'kw': 'm2'}),
    df_elevator[['kw']].rename(columns={'kw': 'diantee'}),
], axis=1)
df_weather = weather.set_index('ObsTime')
df_weather.index= pd.to_datetime(df_weather.index)
df_ac.index= pd.to_datetime(df_ac.index)


putong_all_weather = pd.concat([putong_all, df_weather], axis=1)
putong_all_weather

/var/folders/j_/8fyp4pn14t305lghbc1n8n3w0000gn/T/ipykernel_11065/1839162025.py:13: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  putong_all_weather = pd.concat([putong_all, df_weather], axis=1)


,ac,m1,m2,diantee,StnPres,SeaPres,Temperature,Td dew point,RH,WS,...,Cloud Amount Sat,TxSoil0cm,TxSoil5cm,TxSoil10cm,TxSoil20cm,TxSoil30cm,TxSoil50cm,TxSoil100cm,WSGust,WDGust
2016-01-01 00:00:00,5.74,6.28,0.14,0.08,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-01 01:00:00,5.91,7.00,0.15,0.08,1024.6,1028.3,16.7,11.8,73.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.7,60.0
2016-01-01 02:00:00,6.16,4.56,0.14,0.07,1024.1,1027.8,16.6,12.3,76.0,3.7,...,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,70.0
2016-01-01 03:00:00,6.44,5.80,0.14,0.08,1023.7,1027.4,17.0,12.1,73.0,3.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.7,70.0
2016-01-01 04:00:00,6.65,4.43,0.14,0.08,1023.6,1027.3,16.7,12.5,76.0,3.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.7,40.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 21:00:00,4.30,3.15,0.00,0.07,1013.3,1016.9,17.1,16.0,93.0,2.8,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,6.3,86.0
2025-12-31 22:00:00,4.47,1.64,0.00,0.07,1013.4,1017.0,17.2,15.6,90.0,1.6,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,6.7,97.0
2025-12-31 23:00:00,4.88,2.06,0.00,0.07,1013.6,1017.2,16.9,15.6,92.0,0.9,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,4.1,68.0
2025-12-31 23:59:00,NaN,NaN,NaN,NaN,1013.1,1016.7,16.7,15.6,93.0,0.9,...,NaN,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,-99.5,5.9,94.0


### 2.分出usage type(用電情境)

In [86]:
# 1. 先取出小時資訊
hour = putong_all_weather.index.hour
temp = putong_all_weather['Temperature']

# 2. 定義四個分類的條件
conditions = [
    # 第一類：冷季凌晨用電 (00:00 - 06:00, temp < 18.9)
    (hour >= 0) & (hour < 6) & (temp < 18.9),
    
    # 第二類：冷季下午用電 (12:00 - 18:00, temp < 20.5)
    (hour >= 12) & (hour < 18) & (temp < 20.5),
    
    # 第三類：暖季凌晨用電 (00:00 - 06:00, temp > 28.5)
    (hour >= 0) & (hour < 6) & (temp > 28.5),
    
    # 第四類：暖季下午用電 (12:00 - 18:00, temp > 31.4)
    (hour >= 12) & (hour < 18) & (temp > 31.4)
]

# 3. 定義對應的名稱
choices = [
    '冷季凌晨用電',
    '冷季下午用電',
    '暖季凌晨用電',
    '暖季下午用電'
]

# 4. 建立新欄位 'usage_type'，若不符合以上條件則標記為 '其他'
putong_all_weather['usage_type'] = np.select(conditions, choices, default='其他')



### 3.分出日型與情境

In [87]:
month = putong_all_weather.index.month

# 1. 定義日期清單 (已從您的輸入中去重並整理)
national_holidays = [
    "2016-01-01","2016-02-28","2016-02-29","2016-04-04","2016-04-05","2016-06-09","2016-09-15","2016-10-10",
    "2017-01-01","2017-01-02","2017-02-28","2017-04-04","2017-05-30","2017-10-04","2017-10-10",
    "2018-01-01","2018-02-28","2018-04-04","2018-04-05","2018-06-18","2018-09-24","2018-10-10",
    "2019-01-01","2019-02-28","2019-04-04","2019-04-05","2019-06-07","2019-09-13","2019-10-10",
    "2020-01-01","2020-02-28","2020-04-02","2020-04-03","2020-04-04","2020-04-05","2020-06-25","2020-10-01","2020-10-09","2020-10-10","2020-10-11",
    "2021-01-01","2021-02-28","2021-03-01","2021-04-02","2021-04-03","2021-04-04","2021-04-05","2021-06-14","2021-09-21","2021-10-10","2021-10-11",
    "2022-01-01","2022-02-28","2022-04-02","2022-04-03","2022-04-04","2022-04-05","2022-06-03","2022-09-10","2022-10-10",
    "2023-01-01","2023-01-02","2023-02-28","2023-04-01","2023-04-02","2023-04-03","2023-04-04","2023-04-05","2023-06-22","2023-09-29","2023-10-10",
    "2024-01-01","2024-02-28","2024-04-04","2024-04-05","2024-04-06","2024-04-07","2024-06-10","2024-09-17","2024-10-10",
    "2025-01-01","2025-02-28","2025-04-03","2025-04-04","2025-04-05","2025-04-06","2025-05-31","2025-10-06","2025-10-10"
]
cny_holidays = [
    "2016-02-07","2016-02-08","2016-02-09","2016-02-10","2016-02-11","2016-02-12",
    "2017-01-27","2017-01-28","2017-01-29","2017-01-30","2017-01-31","2017-02-01",
    "2018-02-15","2018-02-16","2018-02-17","2018-02-18","2018-02-19","2018-02-20",
    "2019-02-04","2019-02-05","2019-02-06","2019-02-07","2019-02-08","2019-02-09",
    "2020-01-24","2020-01-25","2020-01-26","2020-01-27","2020-01-28","2020-01-29",
    "2021-02-11","2021-02-12","2021-02-13","2021-02-14","2021-02-15","2021-02-16",
    "2022-01-31","2022-02-01","2022-02-02","2022-02-03","2022-02-04","2022-02-05",
    "2023-01-21","2023-01-22","2023-01-23","2023-01-24","2023-01-25","2023-01-26",
    "2024-02-09","2024-02-10","2024-02-11","2024-02-12","2024-02-13","2024-02-14",
    "2025-01-28","2025-01-29","2025-01-30","2025-01-31","2025-02-01","2025-02-02"
]

# 2. 轉換為 Set 提升查詢效率
national_holiday_set = set(national_holidays)
cny_holiday_set = set(cny_holidays)

def get_day_type(dt):
    """
    輸入: datetime 物件
    輸出: 0=平日, 1=過年, 2=例假日, 3=寒暑假
    """
    date_str = dt.strftime('%Y-%m-%d')
    
    # 優先級：過年 > 寒暑假 > 週末與國定假日 > 平日
    if date_str in cny_holiday_set:
        return 1 # 過年
    elif dt.month in [1, 2, 7, 8]:
        return 3  # 寒暑假
    elif date_str in national_holiday_set or dt.weekday() >= 5:
        return 2 # 週末與國定假日
    else :
        return 0

# 3. 使用範例 (假設您有一個 DataFrame 叫 df，裡面有 'datetime' 欄位)
putong_all_weather['DayType'] = putong_all_weather.index.map(get_day_type)
putong_all_weather['DayType'] 

2016-01-01 00:00:00    3
2016-01-01 01:00:00    3
2016-01-01 02:00:00    3
2016-01-01 03:00:00    3
2016-01-01 04:00:00    3
                      ..
2025-12-31 21:00:00    0
2025-12-31 22:00:00    0
2025-12-31 23:00:00    0
2025-12-31 23:59:00    0
2026-01-01 00:00:00    3
Name: DayType, Length: 91326, dtype: int64

In [88]:
# 結合usage type和daytype分出十六種類

usage_map = {
    '冷季凌晨用電': 'a',
    '冷季下午用電': 'b',
    '暖季凌晨用電': 'c',
    '暖季下午用電': 'd'
}

daytype_map = {
    0: '0',
    1: '1',
    2: '2',
    3: '3'
}

# 建立新欄位：直接將兩個轉換後的字串相加
# 如果 usage 或 daytype 不在字典中，會回傳 NaN
putong_all_weather['type_code'] = putong_all_weather['usage_type'].map(usage_map) + putong_all_weather['DayType'].map(daytype_map)

### 4.計算分項用電

In [89]:
# 普通各電表加總得出總用電量
putong_all_weather['kWh'] = putong_all_weather[['ac', 'm1', 'm2', 'diantee']].sum(axis=1)

In [90]:
# 計算十六個用電情境+日型的用電量
kWh_sum = putong_all_weather.groupby('type_code')['kWh'].sum()

# 取出用電量的值
a1 = kWh_sum.get('a0', 0)
a2 = kWh_sum.get('a1', 0)
a3 = kWh_sum.get('a2', 0)
a4 = kWh_sum.get('a3', 0)

b1 = kWh_sum.get('b0', 0)
b2 = kWh_sum.get('b1', 0)
b3 = kWh_sum.get('b2', 0)
b4 = kWh_sum.get('b3', 0)

c1 = kWh_sum.get('c0', 0)
c2 = kWh_sum.get('c1', 0)
c3 = kWh_sum.get('c2', 0)
c4 = kWh_sum.get('c3', 0)

d1 = kWh_sum.get('d0', 0)
d2 = kWh_sum.get('d1', 0)
d3 = kWh_sum.get('d2', 0)
d4 = kWh_sum.get('d3', 0)

In [91]:
kWh_sum

type_code
a0     19469.290944
a1      2503.850000
a2      7165.825000
a3     30613.083714
b0     85033.960897
b1      1742.537143
b2     15499.488832
b3     57805.021595
c0      4905.150000
c2      2075.370000
c3     17160.915000
d0    231465.609472
d2     32625.702209
d3    200120.316242
Name: kWh, dtype: float64

In [92]:
# A.凌晨用電最低值
A=min(a1, a2, a3, a4)

# B.下午用電最低值
B=min(b1, b2, b3, b4)

# C.設備基本待機用電
X = sorted([a1, a2, a3, a4])[1]
C=max(X-A,0)

# D.設備待機用電增量(暖)
Y=min(c1, c2, c3, c4)
D=max(Y-A-C,0)

# E.上班/研究用電
Z=min(b1, b4)
E=max(Z-B-C,0)

# F.上課用電
F=max(b1-B-C-E,0)

# G.上班/研究空調用電
G=max(d4-B-C-D-E,0)

# H.上課空調用電
H=max(d1-B-C-D-E-F-G,0)

In [93]:
print(A,B,C,D,E,F,G,H)

2503.85 1742.537142857143 4661.975 0 51400.50945238095 27228.93930159841 142315.29464668618 4116.353928579658


### 5.組合分項用電

In [94]:
館舍基礎用電 = A
設備待機用電 = C+D
人員設備使用用電 = E+F
人員空調使用用電 = G+H

In [95]:
print("館舍基礎用電:", 館舍基礎用電)
print("設備待機用電:", 設備待機用電)
print("人員設備使用用電:", 人員設備使用用電)
print("人員空調使用用電:", 人員空調使用用電)

館舍基礎用電: 2503.85
設備待機用電: 4661.975
人員設備使用用電: 78629.44875397936
人員空調使用用電: 146431.64857526583
